In [1]:
# ============================================================
# Temporal Data Preparation - Imports
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# Load Required Datasets
# ============================================================

VALUATIONS_PATH = '../../../data/raw/player_valuations.csv'
APPEARANCES_PATH = '../../../data/raw/appearances.csv'
PLAYERS_PATH = '../../../data/raw/players.csv'

df_valuations = pd.read_csv(VALUATIONS_PATH)
df_appearances = pd.read_csv(APPEARANCES_PATH)
df_players = pd.read_csv(PLAYERS_PATH)

print("Datasets loaded successfully.\n")

print(
    f"Player valuations : {df_valuations.shape}"
)

print(
    f"Appearances       : {df_appearances.shape}"
)

print(
    f"Players           : {df_players.shape}"
)

Datasets loaded successfully.

Player valuations : (640176, 6)
Appearances       : (1884051, 13)
Players           : (47669, 26)


In [3]:
# ============================================================
# Temporal Configuration
# ============================================================

df_valuations['date'] = pd.to_datetime(
    df_valuations['date'],
    errors='coerce'
)

df_appearances['date'] = pd.to_datetime(
    df_appearances['date'],
    errors='coerce'
)

SELECTED_HORIZON_MONTHS = 6
SELECTED_TOLERANCE_DAYS = 45

print("=" * 60)
print("TEMPORAL CONFIGURATION")
print("=" * 60)

print(
    f"Forecast horizon : "
    f"{SELECTED_HORIZON_MONTHS} months"
)

print(
    f"Target tolerance : "
    f"±{SELECTED_TOLERANCE_DAYS} days"
)

print("\nValuation range:")
print(
    df_valuations['date'].min(),
    "→",
    df_valuations['date'].max()
)

print("\nAppearance range:")
print(
    df_appearances['date'].min(),
    "→",
    df_appearances['date'].max()
)

TEMPORAL CONFIGURATION
Forecast horizon : 6 months
Target tolerance : ±45 days

Valuation range:
2000-01-20 00:00:00 → 2026-05-18 00:00:00

Appearance range:
2012-07-03 00:00:00 → 2026-05-18 00:00:00


In [4]:
# ============================================================
# Create Candidate Temporal Snapshots
# ============================================================

first_appearance = (
    df_appearances
    .groupby('player_id')['date']
    .min()
    .reset_index()
    .rename(
        columns={
            'date': 'first_appearance_date'
        }
    )
)

snapshots = (
    df_valuations
    .merge(
        first_appearance,
        on='player_id',
        how='inner'
    )
)

# Keep only valuations occurring on or after
# the player's first recorded appearance
snapshots = snapshots[
    snapshots['date']
    >= snapshots['first_appearance_date']
].copy()

snapshots = snapshots.rename(
    columns={
        'date': 'snapshot_date',
        'market_value_in_eur':
            'current_market_value'
    }
)

print("=" * 60)
print("CANDIDATE TEMPORAL SNAPSHOTS")
print("=" * 60)

print(
    f"Snapshots : {len(snapshots):,}"
)

print(
    f"Players   : "
    f"{snapshots['player_id'].nunique():,}"
)

print("\nSnapshot date range:")

print(
    snapshots['snapshot_date'].min(),
    "→",
    snapshots['snapshot_date'].max()
)

display(
    snapshots[
        [
            'player_id',
            'snapshot_date',
            'current_market_value',
            'current_club_name'
        ]
    ].head()
)

CANDIDATE TEMPORAL SNAPSHOTS
Snapshots : 329,104
Players   : 27,690

Snapshot date range:
2012-07-06 00:00:00 → 2026-05-18 00:00:00


,player_id,snapshot_date,current_market_value,current_club_name
74529,4520,2012-07-06,2500000,Twente Enschede FC
74538,30003,2012-07-06,5000000,Twente Enschede FC
74546,47740,2012-07-06,400000,Twente Enschede FC
74552,60312,2012-07-06,600000,Twente Enschede FC
74565,80073,2012-07-06,1500000,Twente Enschede FC


In [5]:
# ============================================================
# Create Six-Month Future Market Value Target
# ============================================================

# Keep the columns needed from the snapshot table
snapshot_targets = snapshots[
    [
        'player_id',
        'snapshot_date',
        'current_market_value'
    ]
].copy()

# Ideal prediction date = snapshot + 6 months
snapshot_targets['target_date'] = (
    snapshot_targets['snapshot_date']
    + pd.DateOffset(months=SELECTED_HORIZON_MONTHS)
)

# Acceptable target window
snapshot_targets['window_start'] = (
    snapshot_targets['target_date']
    - pd.Timedelta(days=SELECTED_TOLERANCE_DAYS)
)

snapshot_targets['window_end'] = (
    snapshot_targets['target_date']
    + pd.Timedelta(days=SELECTED_TOLERANCE_DAYS)
)

# Prepare all historical valuation records
future_valuations = (
    df_valuations[
        [
            'player_id',
            'date',
            'market_value_in_eur'
        ]
    ]
    .rename(
        columns={
            'date': 'future_date',
            'market_value_in_eur':
                'future_market_value'
        }
    )
    .copy()
)

# Generate candidate future valuations for the same player
target_candidates = snapshot_targets.merge(
    future_valuations,
    on='player_id',
    how='left'
)

# Keep only valuations:
# 1. after the snapshot
# 2. inside the 6-month ±45-day target window
target_candidates = target_candidates[
    (target_candidates['future_date']
        > target_candidates['snapshot_date'])
    &
    (target_candidates['future_date']
        >= target_candidates['window_start'])
    &
    (target_candidates['future_date']
        <= target_candidates['window_end'])
].copy()

# Calculate distance from the ideal 6-month target date
target_candidates['target_distance_days'] = (
    target_candidates['future_date']
    - target_candidates['target_date']
).abs().dt.days

# If several valuations exist inside the window,
# select the one closest to the ideal target date
target_candidates = (
    target_candidates
    .sort_values(
        [
            'player_id',
            'snapshot_date',
            'target_distance_days'
        ]
    )
    .drop_duplicates(
        subset=[
            'player_id',
            'snapshot_date'
        ],
        keep='first'
    )
)

print("=" * 70)
print("FUTURE TARGET CREATION")
print("=" * 70)

print(
    f"Candidate snapshots : "
    f"{len(snapshot_targets):,}"
)

print(
    f"Matched targets     : "
    f"{len(target_candidates):,}"
)

print(
    f"Players             : "
    f"{target_candidates['player_id'].nunique():,}"
)

print(
    f"Coverage            : "
    f"{len(target_candidates) / len(snapshot_targets) * 100:.2f}%"
)

print(
    f"Median target distance: "
    f"{target_candidates['target_distance_days'].median():.1f} days"
)

FUTURE TARGET CREATION
Candidate snapshots : 329,104
Matched targets     : 213,905
Players             : 24,275
Coverage            : 65.00%
Median target distance: 18.0 days


In [6]:
# ============================================================
# Validate Future Market Value Targets
# ============================================================

future_after_snapshot = (
    target_candidates['future_date']
    > target_candidates['snapshot_date']
).all()

inside_tolerance = (
    target_candidates['target_distance_days']
    <= SELECTED_TOLERANCE_DAYS
).all()

missing_targets = (
    target_candidates['future_market_value']
    .isna()
    .sum()
)

duplicate_snapshots = (
    target_candidates
    .duplicated(
        subset=[
            'player_id',
            'snapshot_date'
        ]
    )
    .sum()
)

print("=" * 70)
print("FUTURE TARGET VALIDATION")
print("=" * 70)

print(
    f"All future dates after snapshot : "
    f"{future_after_snapshot}"
)

print(
    f"All targets within tolerance    : "
    f"{inside_tolerance}"
)

print(
    f"Missing future market values    : "
    f"{missing_targets:,}"
)

print(
    f"Duplicate player-date snapshots : "
    f"{duplicate_snapshots:,}"
)

print("\nTarget distance statistics:")

display(
    target_candidates[
        'target_distance_days'
    ].describe()
)

FUTURE TARGET VALIDATION
All future dates after snapshot : True
All targets within tolerance    : True
Missing future market values    : 0
Duplicate player-date snapshots : 0

Target distance statistics:


count   213,905.00
mean         19.19
std          12.67
min           0.00
25%           8.00
50%          18.00
75%          29.00
max          45.00
Name: target_distance_days, dtype: float64

In [7]:
# ============================================================
# Inspect Temporal Target Examples
# ============================================================

target_examples = target_candidates[
    [
        'player_id',
        'snapshot_date',
        'current_market_value',
        'target_date',
        'future_date',
        'future_market_value',
        'target_distance_days'
    ]
].copy()

display(
    target_examples.head(10)
)

,player_id,snapshot_date,current_market_value,target_date,future_date,future_market_value,target_distance_days
95610,10,2013-01-14,4000000,2013-07-14,2013-06-19,2000000,25
225375,10,2013-06-19,2000000,2013-12-19,2014-01-07,1000000,19
360212,10,2014-01-07,1000000,2014-07-07,2014-07-07,1000000,0
558100,10,2014-07-07,1000000,2015-01-07,2015-01-07,1000000,0
736003,10,2015-01-07,1000000,2015-07-07,2015-07-01,1000000,6
1006609,10,2015-07-01,1000000,2016-01-01,2016-01-04,1000000,3
79350,26,2013-01-13,4000000,2013-07-13,2013-06-30,5000000,13
558840,26,2014-07-10,5000000,2015-01-10,2015-02-04,3000000,25
818169,26,2015-02-04,3000000,2015-08-04,2015-07-01,2000000,34
1006632,26,2015-07-01,2000000,2016-01-01,2016-02-15,1000000,45


In [8]:
# ============================================================
# Inspect Player Attributes for Temporal Safety
# ============================================================

print("=" * 70)
print("PLAYER ATTRIBUTE AUDIT")
print("=" * 70)

print("\nAvailable player columns:")
print(df_players.columns.tolist())

player_audit_columns = [
    col for col in [
        'player_id',
        'name',
        'date_of_birth',
        'position',
        'sub_position',
        'foot',
        'height_in_cm',
        'country_of_citizenship',
        'current_club_id',
        'current_club_name',
        'highest_market_value_in_eur'
    ]
    if col in df_players.columns
]

print("\nSelected columns for inspection:")

display(
    df_players[
        player_audit_columns
    ].head(10)
)

PLAYER ATTRIBUTE AUDIT

Available player columns:
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

Selected columns for inspection:


,player_id,name,date_of_birth,position,sub_position,foot,height_in_cm,country_of_citizenship,current_club_id,current_club_name,highest_market_value_in_eur
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,Centre-Forward,right,184.00,Germany,398,Società Sportiva Lazio S.p.A.,"30,000,000.00"
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,Goalkeeper,left,190.00,Germany,16,Borussia Dortmund,"8,000,000.00"
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,Centre-Forward,NaN,NaN,Bulgaria,1091,Panthessalonikios Athlitikos Omilos Konstantin...,"34,500,000.00"
3,77,Lúcio,1978-05-08 00:00:00,Defender,Centre-Back,NaN,NaN,Brazil,506,Juventus Football Club,"24,500,000.00"
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,Goalkeeper,right,194.00,Germany,27,FC Bayern München,"3,000,000.00"
5,109,Dedê,1978-04-18 00:00:00,Defender,Left-Back,NaN,NaN,Brazil,825,Eskisehirspor,"9,500,000.00"
6,123,Christoph Metzelder,1980-11-05 00:00:00,Defender,Centre-Back,NaN,NaN,Germany,33,FC Schalke 04,"9,500,000.00"
7,132,Tomas Rosicky,1980-10-04 00:00:00,Midfield,Attacking Midfield,both,179.00,Czech Republic,11,Arsenal Football Club,"17,500,000.00"
8,162,Marc Ziegler,1976-06-13 00:00:00,Goalkeeper,Goalkeeper,right,193.00,Germany,79,Verein für Bewegungsspiele Stuttgart 1893,"1,250,000.00"
9,215,Roque Santa Cruz,1981-08-16 00:00:00,Attack,Centre-Forward,right,193.00,Paraguay,1084,Málaga CF,"12,000,000.00"
